In [ ]:
# Compares the old and new VideoMAE checkpoints on the same 335 test videos.

# Comparaison — ancien vs nouveau VideoMAE

Ce notebook compare les deux checkpoints sur **les mêmes 335 vidéos exactes**
que le test standardisé historique.

Contrairement à la première version, il ne recrée pas un nouveau split
aléatoire. Il lit directement les identifiants présents dans :

- `videomae_test_predictions.csv` ;
- `videomae_validation_predictions.csv`.

## Inputs Kaggle à ajouter

- `Training2`
- `VideoMAE Training2 Finetuned Results`
- `videomae-final-rhad-results`

Les identifiants des jeux historiques de validation et de test sont intégrés
directement dans ce notebook. Aucun CSV supplémentaire n'est requis.

## Résultats produits

- comparaison globale avant/après ;
- Accuracy Top-1 et Top-5 ;
- Macro Precision, Recall et F1 ;
- résultats par classe ;
- résultats spécifiques à `objects_falling` ;
- matrices de confusion brutes et normalisées ;
- prédictions du nouveau checkpoint ;
- liste des erreurs corrigées et des régressions ;
- archive ZIP finale.

Active **GPU T4 x2**, puis lance **Run All**.


In [ ]:
# VIDEOMAE — CELLULE 2
# Création d'un environnement Python 3.10 isolé

from pathlib import Path
import shutil
import subprocess
import sys

ENV_DIR = Path("/kaggle/working/vmae_env")

# Installer uv, utilisé pour créer rapidement l'environnement
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "uv",
    ],
    check=True,
)

UV = shutil.which("uv")

if UV is None:
    raise FileNotFoundError(
        "La commande uv n'a pas été trouvée."
    )

print("uv trouvé :", UV)

# Supprimer uniquement un éventuel ancien environnement VideoMAE
shutil.rmtree(
    ENV_DIR,
    ignore_errors=True,
)

# Installer Python 3.10
subprocess.run(
    [
        UV,
        "python",
        "install",
        "3.10",
    ],
    check=True,
)

# Créer l'environnement isolé
subprocess.run(
    [
        UV,
        "venv",
        "--python",
        "3.10",
        str(ENV_DIR),
    ],
    check=True,
)

PY = str(ENV_DIR / "bin" / "python")

assert Path(PY).exists(), (
    f"Python absent : {PY}"
)

# Vérification
subprocess.run(
    [
        PY,
        "--version",
    ],
    check=True,
)

print("Environnement créé :", ENV_DIR)
print("Python utilisé :", PY)
print("ENVIRONNEMENT VIDEOMAE VALIDÉ")

In [ ]:
# VIDEOMAE — CELLULE 3
# Installation de PyTorch et des dépendances VideoMAEv2

from pathlib import Path
import os
import shutil
import subprocess

ENV_DIR = Path("/kaggle/working/vmae_env")
PY = str(ENV_DIR / "bin" / "python")

assert Path(PY).exists(), (
    f"Python de l'environnement absent : {PY}"
)

UV = shutil.which("uv")

if UV is None:
    raise FileNotFoundError(
        "La commande uv est introuvable."
    )

print("=" * 70)
print("1. INSTALLATION DE PYTORCH + CUDA 11.8")
print("=" * 70)

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--python",
        PY,
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
        "torch==2.1.2",
        "torchvision==0.16.2",
        "torchaudio==2.1.2",
    ],
    check=True,
)

print("\n" + "=" * 70)
print("2. INSTALLATION DES DÉPENDANCES VIDEOMAE")
print("=" * 70)

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--python",
        PY,

        # NumPy 1.x pour éviter les incompatibilités
        # avec les anciennes bibliothèques VideoMAE
        "numpy==1.26.4",

        # Version recommandée par VideoMAEv2
        "timm==0.4.12",

        # Lecture et traitement vidéo
        "decord==0.6.0",
        "av",
        "opencv-python-headless",
        "Pillow",

        # Calcul scientifique
        "einops",
        "scipy",
        "pandas",

        # Logs et résultats
        "tensorboardX",
        "matplotlib",
        "scikit-learn",

        # Téléchargement du modèle préentraîné
        "huggingface_hub",
    ],
    check=True,
)

print("\n" + "=" * 70)
print("3. VÉRIFICATION DES INSTALLATIONS")
print("=" * 70)

verification_code = r"""
import sys

import torch
import torchvision
import timm
import numpy
import decord
import av
import cv2
import einops
import scipy
import pandas
import tensorboardX
import sklearn
import huggingface_hub

print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Torchvision  :", torchvision.__version__)
print("CUDA PyTorch :", torch.version.cuda)
print("CUDA dispo   :", torch.cuda.is_available())
print("Nombre GPU   :", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(
        f"GPU {index}       :",
        torch.cuda.get_device_name(index)
    )

print("timm          :", timm.__version__)
print("NumPy         :", numpy.__version__)
print("decord        :", decord.__version__)
print("PyAV          :", av.__version__)
print("OpenCV        :", cv2.__version__)
print("SciPy         :", scipy.__version__)
print("Pandas        :", pandas.__version__)

# Petit calcul GPU réel
if not torch.cuda.is_available():
    raise RuntimeError("CUDA n'est pas disponible dans l'environnement.")

x = torch.randn(
    128,
    128,
    device="cuda",
)

y = x @ x.T

assert y.shape == (128, 128)
assert torch.isfinite(y).all()

print("Test calcul GPU : VALIDÉ")
print("DÉPENDANCES VIDEOMAE VALIDÉES")
"""

subprocess.run(
    [
        PY,
        "-c",
        verification_code,
    ],
    check=True,
)

In [ ]:
# VIDEOMAE — CELLULE 4A
# Télécharger exactement la version utilisée du dépôt

from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path("/kaggle/working/VideoMAEv2")
COMMIT = "29eab1e8a588d1b3ec0cdec7b03a86cca491b74b"

shutil.rmtree(REPO_DIR, ignore_errors=True)

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/OpenGVLab/VideoMAEv2.git",
        str(REPO_DIR),
    ],
    check=True,
)

subprocess.run(
    ["git", "checkout", COMMIT],
    cwd=str(REPO_DIR),
    check=True,
)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=str(REPO_DIR),
    text=True,
).strip()

print("Commit téléchargé :", actual_commit)
assert actual_commit == COMMIT
print("DÉPÔT VIDEOMAEV2 VALIDÉ")


In [ ]:
# VIDEOMAE — CELLULE 4B
# Vérification du dépôt et création du modèle

from pathlib import Path
import subprocess

REPO_DIR = Path("/kaggle/working/VideoMAEv2")
PY = "/kaggle/working/vmae_env/bin/python"

assert REPO_DIR.exists(), (
    f"Dépôt absent : {REPO_DIR}"
)

assert Path(PY).exists(), (
    f"Environnement Python absent : {PY}"
)

# ============================================================
# 1. Vérification des vrais chemins du dépôt
# ============================================================

required_files = [
    REPO_DIR / "run_class_finetuning.py",
    REPO_DIR / "dataset" / "build.py",
    REPO_DIR / "dataset" / "datasets.py",
    REPO_DIR / "models" / "__init__.py",
    REPO_DIR / "models" / "modeling_finetune.py",
    REPO_DIR / "utils.py",
]

print("=" * 70)
print("1. FICHIERS ESSENTIELS")
print("=" * 70)

for file_path in required_files:
    exists = file_path.exists()

    print(
        file_path.relative_to(REPO_DIR),
        "->",
        exists,
    )

    assert exists, (
        f"Fichier essentiel absent : {file_path}"
    )

# ============================================================
# 2. Diagnostic du script d'entraînement
# ============================================================

training_script = (
    REPO_DIR / "run_class_finetuning.py"
).read_text(
    encoding="utf-8"
)

dataset_build = (
    REPO_DIR / "dataset" / "build.py"
).read_text(
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("2. DIAGNOSTIC DU SCRIPT")
print("=" * 70)

print(
    "Import obligatoire de DeepSpeed :",
    "import deepspeed" in training_script,
)

print(
    "Option --finetune présente :",
    "--finetune" in training_script,
)

print(
    "Option --resume présente :",
    "--resume" in training_script,
)

print(
    "Option --save_ckpt présente :",
    "--save_ckpt" in training_script,
)

print(
    "Dataset custom déjà supporté :",
    "'custom'" in dataset_build
    or '"custom"' in dataset_build,
)

# ============================================================
# 3. Test de création de VideoMAE
# ============================================================

print("\n" + "=" * 70)
print("3. TEST DE CRÉATION DU MODÈLE")
print("=" * 70)

test_code = r"""
import torch
import timm

# Cet import enregistre les modèles VideoMAE dans timm
import models

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=12,
    all_frames=16,
    tubelet_size=2,
    drop_path_rate=0.1,
    use_mean_pooling=True,
    init_scale=0.001,
)

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

classifier = model.get_classifier()

print("Modèle créé :", model.__class__.__name__)
print("Nombre de paramètres :", f"{number_of_parameters:,}")
print(
    "Nombre de classes :",
    getattr(classifier, "out_features", "non détecté"),
)
print("Frames :", 16)
print("Tubelet size :", 2)

assert getattr(classifier, "out_features", None) == 12

print("IMPORT ET CRÉATION VIDEOMAE VALIDÉS")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("Messages système / erreurs :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test de création VideoMAE a échoué."
    )

print("\nDÉPÔT VIDEOMAEV2 VALIDÉ")

In [ ]:
# VIDEOMAE — CELLULE 5
# Adaptation du dépôt à RHAD, sans DeepSpeed

from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path("/kaggle/working/VideoMAEv2")
PY = "/kaggle/working/vmae_env/bin/python"

RUN_FILE = REPO_DIR / "run_class_finetuning.py"
BUILD_FILE = REPO_DIR / "dataset" / "build.py"

assert RUN_FILE.exists(), RUN_FILE
assert BUILD_FILE.exists(), BUILD_FILE
assert Path(PY).exists(), PY

# ============================================================
# 1. SAUVEGARDES DU CODE ORIGINAL
# ============================================================

for file_path in [RUN_FILE, BUILD_FILE]:
    backup_path = file_path.with_suffix(
        file_path.suffix + ".original"
    )

    if not backup_path.exists():
        shutil.copy2(
            file_path,
            backup_path,
        )

        print(
            "Sauvegarde créée :",
            backup_path,
        )
    else:
        print(
            "Sauvegarde déjà présente :",
            backup_path,
        )

# ============================================================
# 2. RENDRE DEEPSPEED OPTIONNEL
# ============================================================

run_text = RUN_FILE.read_text(
    encoding="utf-8"
)

if (
    "try:\n    import deepspeed"
    not in run_text
):
    assert "import deepspeed" in run_text

    run_text = run_text.replace(
        "import deepspeed",
        """try:
    import deepspeed
except ImportError:
    deepspeed = None""",
        1,
    )

# Sécuriser l'option --enable_deepspeed
old_deepspeed_block = """    if known_args.enable_deepspeed:
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""

new_deepspeed_block = """    if known_args.enable_deepspeed:
        if deepspeed is None:
            raise ImportError(
                "DeepSpeed a été demandé mais il n'est pas installé."
            )
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""

if old_deepspeed_block in run_text:
    run_text = run_text.replace(
        old_deepspeed_block,
        new_deepspeed_block,
        1,
    )

# ============================================================
# 3. AJOUTER RHAD AUX DATASETS AUTORISÉS
# ============================================================

data_argument_start = run_text.find(
    "parser.add_argument(\n"
    "        '--data_set'"
)

assert data_argument_start != -1, (
    "Argument --data_set introuvable."
)

data_argument_end = run_text.find(
    "type=str",
    data_argument_start,
)

assert data_argument_end != -1

data_argument_block = run_text[
    data_argument_start:data_argument_end
]

if "'RHAD'" not in data_argument_block:
    assert "'MIT'" in data_argument_block

    patched_argument_block = (
        data_argument_block.replace(
            "'MIT'",
            "'MIT', 'RHAD'",
            1,
        )
    )

    run_text = (
        run_text[:data_argument_start]
        + patched_argument_block
        + run_text[data_argument_end:]
    )

RUN_FILE.write_text(
    run_text,
    encoding="utf-8",
)

# ============================================================
# 4. UTILISER TEST.CSV POUR LE VRAI TEST
# ============================================================

build_text = BUILD_FILE.read_text(
    encoding="utf-8"
)

old_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'val.csv')
"""

new_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'test.csv')
"""

if old_test_block in build_text:
    build_text = build_text.replace(
        old_test_block,
        new_test_block,
        1,
    )

# ============================================================
# 5. AJOUTER LE DATASET RHAD : 12 CLASSES
# ============================================================

rhad_branch = """
    elif args.data_set == 'RHAD':
        dataset = VideoClsDataset(
            anno_path=anno_path,
            data_root=args.data_root,
            mode=mode,
            clip_len=args.num_frames,
            frame_sample_rate=args.sampling_rate,
            num_segment=1,
            test_num_segment=args.test_num_segment,
            test_num_crop=args.test_num_crop,
            num_crop=1 if not test_mode else 3,
            keep_aspect_ratio=True,
            crop_size=args.input_size,
            short_side_size=args.short_side_size,
            new_height=256,
            new_width=320,
            sparse_sample=False,
            args=args,
        )
        nb_classes = 12
"""

unsupported_block = """    else:
        raise NotImplementedError('Unsupported Dataset')
"""

if "elif args.data_set == 'RHAD':" not in build_text:
    assert unsupported_block in build_text, (
        "Bloc Unsupported Dataset introuvable."
    )

    build_text = build_text.replace(
        unsupported_block,
        rhad_branch + "\n" + unsupported_block,
        1,
    )

BUILD_FILE.write_text(
    build_text,
    encoding="utf-8",
)

# ============================================================
# 6. VÉRIFIER LES PATCHS
# ============================================================

final_run_text = RUN_FILE.read_text(
    encoding="utf-8"
)

final_build_text = BUILD_FILE.read_text(
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("VÉRIFICATION DES MODIFICATIONS")
print("=" * 70)

checks = {
    "DeepSpeed optionnel":
        "except ImportError" in final_run_text,

    "RHAD autorisé dans les arguments":
        "'RHAD'" in final_run_text,

    "RHAD présent dans build.py":
        "elif args.data_set == 'RHAD':"
        in final_build_text,

    "RHAD possède 12 classes":
        "nb_classes = 12"
        in final_build_text,

    "Le test utilise test.csv":
        "anno_path = os.path.join("
        "args.data_path, 'test.csv')"
        in final_build_text,
}

for name, result in checks.items():
    print(f"{name:35s} :", result)
    assert result, name

# ============================================================
# 7. VÉRIFICATION DE LA SYNTAXE PYTHON
# ============================================================

subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(RUN_FILE),
        str(BUILD_FILE),
    ],
    check=True,
)

# Vérifier que les imports fonctionnent sans DeepSpeed
test_code = r"""
import run_class_finetuning
from dataset import build_dataset

print("Import run_class_finetuning : OK")
print("Import build_dataset : OK")
print("PATCH VIDEOMAE RHAD VALIDÉ")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)

print("\nSortie du test :")
print(result.stdout)

if result.stderr:
    print("Messages système :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test des fichiers modifiés a échoué."
    )

print("\nADAPTATION DU DÉPÔT TERMINÉE")

In [ ]:
# VIDEOMAE — CELLULE 9
# Conserver uniquement checkpoint-latest et checkpoint-best

from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"

UTILS_FILE = Path(
    "/kaggle/working/VideoMAEv2/utils.py"
)

assert Path(PY).exists(), PY
assert UTILS_FILE.exists(), UTILS_FILE

# ============================================================
# 1. SAUVEGARDE DU FICHIER ORIGINAL
# ============================================================

BACKUP_FILE = UTILS_FILE.with_suffix(
    ".py.original"
)

if not BACKUP_FILE.exists():
    shutil.copy2(
        UTILS_FILE,
        BACKUP_FILE,
    )
    print("Sauvegarde créée :", BACKUP_FILE)
else:
    print("Sauvegarde déjà présente :", BACKUP_FILE)

# ============================================================
# 2. MODIFICATION DU NOM DES CHECKPOINTS
# ============================================================

text = UTILS_FILE.read_text(
    encoding="utf-8"
)

old_code = """    if loss_scaler is not None:
        checkpoint_paths = [output_dir / ('checkpoint-%s.pth' % epoch_name)]
"""

new_code = """    if loss_scaler is not None:
        # Le meilleur modèle reste dans checkpoint-best.pth.
        # Tous les checkpoints périodiques écrasent checkpoint-latest.pth
        # afin de ne pas créer des dizaines de fichiers volumineux.
        if epoch_name == "best":
            checkpoint_filename = "checkpoint-best.pth"
        else:
            checkpoint_filename = "checkpoint-latest.pth"

        checkpoint_paths = [
            output_dir / checkpoint_filename
        ]
"""

if old_code in text:
    text = text.replace(
        old_code,
        new_code,
        1,
    )
    UTILS_FILE.write_text(
        text,
        encoding="utf-8",
    )
    print("Patch appliqué.")
elif new_code in text:
    print("Patch déjà appliqué.")
else:
    raise RuntimeError(
        "Le bloc de sauvegarde attendu n'a pas été trouvé."
    )

# ============================================================
# 3. VÉRIFICATIONS
# ============================================================

final_text = UTILS_FILE.read_text(
    encoding="utf-8"
)

checks = {
    "checkpoint-latest présent":
        'checkpoint_filename = "checkpoint-latest.pth"'
        in final_text,

    "checkpoint-best présent":
        'checkpoint_filename = "checkpoint-best.pth"'
        in final_text,

    "ancien fichier numérique supprimé":
        "checkpoint_paths = [output_dir / "
        "('checkpoint-%s.pth' % epoch_name)]"
        not in final_text,
}

print("\n" + "=" * 70)
print("VÉRIFICATION DU PATCH")
print("=" * 70)

for name, result in checks.items():
    print(f"{name:40s} :", result)
    assert result, name

# Vérification de la syntaxe
subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(UTILS_FILE),
    ],
    check=True,
)

# Vérification par import
test_code = r"""
import utils

print("Import utils : OK")
print("GESTION DES CHECKPOINTS VALIDÉE")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd="/kaggle/working/VideoMAEv2",
    text=True,
    capture_output=True,
)

print("\n" + result.stdout)

if result.stderr:
    print("Messages système :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test du patch checkpoints a échoué."
    )

print("\nPRÉPARATION DES CHECKPOINTS TERMINÉE")

In [ ]:
# VIDEOMAE — CELLULE 10A
# Corriger sampler_test

from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"
RUN_FILE = Path(
    "/kaggle/working/VideoMAEv2/run_class_finetuning.py"
)

assert RUN_FILE.exists()
assert Path(PY).exists()

backup = RUN_FILE.with_name(
    "run_class_finetuning.py.before_sampler_patch"
)

if not backup.exists():
    shutil.copy2(RUN_FILE, backup)

text = RUN_FILE.read_text(encoding="utf-8")

old_block = """    else:
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)
"""

new_block = """    else:
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)
        sampler_test = torch.utils.data.DistributedSampler(
            dataset_test,
            num_replicas=num_tasks,
            rank=global_rank,
            shuffle=False,
        )
"""

start = text.index("    if args.dist_eval:")
end = text.index(
    "    if global_rank == 0 and args.log_dir is not None:",
    start,
)
section = text[start:end]
else_section = section.split("    else:", 1)[-1]

if "sampler_test =" not in else_section:
    assert old_block in section
    section = section.replace(old_block, new_block, 1)
    text = text[:start] + section + text[end:]
    RUN_FILE.write_text(text, encoding="utf-8")
    print("Patch sampler_test appliqué.")
else:
    print("Patch sampler_test déjà présent.")

final_text = RUN_FILE.read_text(encoding="utf-8")
start = final_text.index("    if args.dist_eval:")
end = final_text.index(
    "    if global_rank == 0 and args.log_dir is not None:",
    start,
)
final_section = final_text[start:end]

assert "sampler_test =" in final_section.split("    else:", 1)[-1]

subprocess.run(
    [PY, "-m", "py_compile", str(RUN_FILE)],
    check=True,
)

print("SAMPLER_TEST VALIDÉ")


In [ ]:
# VIDEOMAE — CELLULE 10B
# Remplacer le bfloat16 incompatible avec les T4 par du float16

from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"

ENGINE_FILE = Path(
    "/kaggle/working/VideoMAEv2/"
    "engine_for_finetuning.py"
)

assert Path(PY).exists(), PY
assert ENGINE_FILE.exists(), ENGINE_FILE

# ============================================================
# 1. SAUVEGARDE
# ============================================================

BACKUP_FILE = ENGINE_FILE.with_name(
    "engine_for_finetuning.py.before_fp16_patch"
)

if not BACKUP_FILE.exists():
    shutil.copy2(
        ENGINE_FILE,
        BACKUP_FILE,
    )
    print("Sauvegarde créée :", BACKUP_FILE)
else:
    print("Sauvegarde déjà présente :", BACKUP_FILE)

# ============================================================
# 2. INSPECTION AVANT MODIFICATION
# ============================================================

text = ENGINE_FILE.read_text(
    encoding="utf-8"
)

bf16_count_before = text.count(
    "torch.bfloat16"
)

print(
    "Occurrences torch.bfloat16 avant patch :",
    bf16_count_before,
)

if bf16_count_before == 0:
    print(
        "Aucune occurrence BF16 : "
        "le patch semble déjà appliqué."
    )
else:
    # Le fichier concerne uniquement le fine-tuning.
    # Toutes ses opérations AMP doivent utiliser FP16 sur T4.
    text = text.replace(
        "torch.bfloat16",
        "torch.float16",
    )

    ENGINE_FILE.write_text(
        text,
        encoding="utf-8",
    )

    print(
        "Toutes les occurrences BF16 ont été "
        "remplacées par FP16."
    )

# ============================================================
# 3. VÉRIFICATION
# ============================================================

final_text = ENGINE_FILE.read_text(
    encoding="utf-8"
)

bf16_count_after = final_text.count(
    "torch.bfloat16"
)

fp16_count = final_text.count(
    "torch.float16"
)

print("\n" + "=" * 70)
print("VÉRIFICATION DU PATCH")
print("=" * 70)

print(
    "Occurrences torch.bfloat16 après patch :",
    bf16_count_after,
)

print(
    "Occurrences torch.float16 :",
    fp16_count,
)

assert bf16_count_after == 0, (
    "Il reste encore une opération en bfloat16."
)

assert fp16_count >= 1, (
    "Aucune opération float16 n'a été trouvée."
)

# Vérifier la syntaxe Python
subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(ENGINE_FILE),
    ],
    check=True,
)

# ============================================================
# 4. TEST AMP RÉEL SUR LE T4
# ============================================================

test_code = r"""
import torch

assert torch.cuda.is_available()

x = torch.randn(
    256,
    256,
    device="cuda",
)

with torch.cuda.amp.autocast(
    dtype=torch.float16
):
    y = x @ x.T

print("GPU :", torch.cuda.get_device_name(0))
print("Type de sortie :", y.dtype)
print("Forme :", tuple(y.shape))
print("Valeurs finies :", torch.isfinite(y).all().item())

assert y.dtype == torch.float16
assert torch.isfinite(y).all()

print("AMP FLOAT16 VALIDÉ SUR TESLA T4")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    text=True,
    capture_output=True,
)

print("\n" + result.stdout)

if result.stderr:
    print("Messages système :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test FP16 sur le GPU a échoué."
    )

print("\nPATCH FP16 VIDEOMAE VALIDÉ")

In [ ]:
# ============================================================
# DÉTECTER LES INPUTS : TRAINING2 + DEUX CHECKPOINTS
# ============================================================

from pathlib import Path
import torch

KAGGLE_INPUT = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass"
)

# Dataset Training2
training_candidates = [
    path
    for path in KAGGLE_INPUT.rglob("Training")
    if path.is_dir() and path.parent.name.lower() == "training2"
]

print("Candidats Training2/Training :")
for path in training_candidates:
    print(" -", path)

assert len(training_candidates) == 1, (
    f"Un seul Training2/Training attendu, trouvé : {len(training_candidates)}"
)

TRAINING2_ROOT = training_candidates[0]

# Tous les checkpoint-best
checkpoint_candidates = [
    path
    for path in KAGGLE_INPUT.rglob("checkpoint-best.pth")
    if path.is_file()
]

print("\nCheckpoints trouvés :")
for path in checkpoint_candidates:
    print(" -", path)

old_candidates = [
    path for path in checkpoint_candidates
    if "videomae-final-rhad-results" in str(path)
]

new_candidates = [
    path for path in checkpoint_candidates
    if "videomae-training2-finetuned-results" in str(path)
]

assert len(old_candidates) == 1, (
    "Ancien checkpoint non trouvé de manière unique."
)

assert len(new_candidates) == 1, (
    "Nouveau checkpoint non trouvé de manière unique."
)

OLD_CHECKPOINT = old_candidates[0]
NEW_CHECKPOINT = new_candidates[0]

for checkpoint_path in [OLD_CHECKPOINT, NEW_CHECKPOINT]:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    assert "model" in checkpoint
    assert tuple(checkpoint["model"]["head.weight"].shape) == (12, 768)

print("\n" + "=" * 70)
print("CONFIGURATION")
print("=" * 70)
print("Training2        :", TRAINING2_ROOT)
print("Ancien checkpoint:", OLD_CHECKPOINT)
print("Nouveau checkpoint:", NEW_CHECKPOINT)


In [ ]:
# ============================================================
# RECONSTRUIRE LE SPLIT À PARTIR DES IDENTIFIANTS HISTORIQUES
# Aucun nouveau split aléatoire n'est créé.
# ============================================================

from pathlib import Path
import csv
import json
import os
import re
import shutil

import pandas as pd

ANNOTATIONS_DIR = Path(
    "/kaggle/working/rhad_comparison_annotations"
)

VIDEO_ROOT = Path(
    "/kaggle/working/rhad_comparison_videos"
)

for directory in [ANNOTATIONS_DIR, VIDEO_ROOT]:
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"
}

CLASS_NAMES = sorted(
    folder.name
    for folder in TRAINING2_ROOT.iterdir()
    if folder.is_dir()
)

assert len(CLASS_NAMES) == 12, CLASS_NAMES

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

# ------------------------------------------------------------
# 1. Charger les deux manifestes historiques intégrés
# ------------------------------------------------------------

embedded_test_records = json.loads(
    r'''[{"video_id":"rhad_00188_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01583_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02516_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_02846_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01700_falling","true_label":5,"true_class":"falling","predicted_label":8,"correct_top1":0},{"video_id":"rhad_00842_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_00518_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00516_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_02027_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_02890_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00662_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00514_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01580_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02433_objects_falling","true_label":8,"true_class":"objects_falling","predicted_label":5,"correct_top1":0},{"video_id":"rhad_01210_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_01493_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_01530_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":10,"correct_top1":0},{"video_id":"rhad_01516_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_01328_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_02069_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_01738_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02885_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02851_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_03001_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00915_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_01845_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_00579_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00451_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01492_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_02546_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_02893_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01849_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_00119_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00782_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_03003_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00859_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_00660_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_03220_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00783_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00769_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_03202_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_02230_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_01532_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":10,"correct_top1":0},{"video_id":"rhad_01943_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_02217_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02186_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_01581_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02205_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02781_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01844_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_03218_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_01077_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_03229_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00843_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_00768_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01865_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01515_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_00284_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00914_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02220_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_03161_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_01735_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_03219_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00196_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00515_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01517_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_01357_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_00406_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02835_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_03231_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_01555_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_03005_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_01356_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_02029_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_01932_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_00198_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01689_falling","true_label":5,"true_class":"falling","predicted_label":8,"correct_top1":0},{"video_id":"rhad_02768_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01531_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":10,"correct_top1":0},{"video_id":"rhad_01941_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_03002_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_02836_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01862_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01037_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_01106_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02519_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_02250_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02852_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01108_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_00064_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01035_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_00428_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00186_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00320_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02767_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02715_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02833_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00841_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_01557_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01078_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_03228_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_01739_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01584_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02779_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00917_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02547_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_01510_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":3,"correct_top1":0},{"video_id":"rhad_01667_falling","true_label":5,"true_class":"falling","predicted_label":8,"correct_top1":0},{"video_id":"rhad_01354_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_01929_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_00622_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01736_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01656_falling","true_label":5,"true_class":"falling","predicted_label":8,"correct_top1":0},{"video_id":"rhad_02251_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02436_objects_falling","true_label":8,"true_class":"objects_falling","predicted_label":5,"correct_top1":0},{"video_id":"rhad_01928_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":2,"correct_top1":0},{"video_id":"rhad_02434_objects_falling","true_label":8,"true_class":"objects_falling","predicted_label":5,"correct_top1":0},{"video_id":"rhad_01850_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01507_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":3,"correct_top1":0},{"video_id":"rhad_03203_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_01847_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02152_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_00862_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02249_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_00318_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02175_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02882_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01360_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":10,"correct_top1":0},{"video_id":"rhad_01938_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_01338_collesion","true_label":3,"true_class":"collesion","predicted_label":2,"correct_top1":0},{"video_id":"rhad_01939_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_01863_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01861_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_00575_child_running","true_label":1,"true_class":"child_running","predicted_label":0,"correct_top1":0},{"video_id":"rhad_00620_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01867_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02834_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02884_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00913_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_01491_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_02031_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_01034_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_03232_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_01202_collesion","true_label":3,"true_class":"collesion","predicted_label":10,"correct_top1":0},{"video_id":"rhad_00351_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00348_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01340_collesion","true_label":3,"true_class":"collesion","predicted_label":2,"correct_top1":0},{"video_id":"rhad_02028_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_01103_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_01330_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_01552_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01854_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_00350_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00780_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00120_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02850_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02891_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02068_normal","true_label":7,"true_class":"normal","predicted_label":1,"correct_top1":0},{"video_id":"rhad_02778_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01209_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_00659_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_02248_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02847_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00858_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02216_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02782_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02209_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_03162_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_03221_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_01212_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_02837_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02769_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02952_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00281_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02141_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_03200_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_02953_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00657_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01199_collesion","true_label":3,"true_class":"collesion","predicted_label":10,"correct_top1":0},{"video_id":"rhad_01529_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":10,"correct_top1":0},{"video_id":"rhad_01869_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01341_collesion","true_label":3,"true_class":"collesion","predicted_label":2,"correct_top1":0},{"video_id":"rhad_02330_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_01495_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_00770_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01512_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":3,"correct_top1":0},{"video_id":"rhad_00784_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00658_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_02253_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_00282_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01339_collesion","true_label":3,"true_class":"collesion","predicted_label":2,"correct_top1":0},{"video_id":"rhad_02889_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00349_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02848_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02951_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00781_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01342_collesion","true_label":3,"true_class":"collesion","predicted_label":2,"correct_top1":0},{"video_id":"rhad_00417_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01036_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02329_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_03166_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_02714_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02130_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_00199_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02208_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_01201_collesion","true_label":3,"true_class":"collesion","predicted_label":10,"correct_top1":0},{"video_id":"rhad_03205_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00857_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02219_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_00439_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00623_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00576_child_running","true_label":1,"true_class":"child_running","predicted_label":0,"correct_top1":0},{"video_id":"rhad_02544_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_01104_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_00494_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00577_child_running","true_label":1,"true_class":"child_running","predicted_label":0,"correct_top1":0},{"video_id":"rhad_00794_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01105_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02888_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02218_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_01846_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02713_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01508_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":3,"correct_top1":0},{"video_id":"rhad_00040_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02333_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_00513_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00122_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00860_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_00844_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_01120_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_02215_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_00493_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_00283_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02204_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02435_objects_falling","true_label":8,"true_class":"objects_falling","predicted_label":5,"correct_top1":0},{"video_id":"rhad_00197_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00280_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01327_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_01870_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_00805_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_02892_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01866_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01554_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_03201_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_02163_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_00319_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01851_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_00619_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_02955_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00845_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02845_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01513_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_02518_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_01200_collesion","true_label":3,"true_class":"collesion","predicted_label":10,"correct_top1":0},{"video_id":"rhad_00201_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_03163_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_02332_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02950_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00124_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_03230_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00065_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01213_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_02070_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_01553_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_00001_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00491_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01211_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_00496_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01860_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_01076_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_01329_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_02521_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_02780_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01737_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02438_objects_falling","true_label":8,"true_class":"objects_falling","predicted_label":5,"correct_top1":0},{"video_id":"rhad_00578_child_running","true_label":1,"true_class":"child_running","predicted_label":0,"correct_top1":0},{"video_id":"rhad_00036_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00779_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_03164_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00492_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01080_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_03004_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00321_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02331_normal","true_label":7,"true_class":"normal","predicted_label":3,"correct_top1":0},{"video_id":"rhad_00816_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_02206_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_00190_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01198_collesion","true_label":3,"true_class":"collesion","predicted_label":10,"correct_top1":0},{"video_id":"rhad_00323_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01355_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_02207_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_03217_sharp_objects","true_label":11,"true_class":"sharp_objects","predicted_label":11,"correct_top1":1},{"video_id":"rhad_00066_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01927_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":2,"correct_top1":0},{"video_id":"rhad_01582_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02711_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00773_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01039_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02517_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_00185_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_01494_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_02712_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01509_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":3,"correct_top1":0},{"video_id":"rhad_00771_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_02766_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_02854_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00187_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02026_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_02067_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_01940_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_02545_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_01514_electrical_appliances","true_label":4,"true_class":"electrical_appliances","predicted_label":4,"correct_top1":1},{"video_id":"rhad_02548_pulling_curtains","true_label":9,"true_class":"pulling_curtains","predicted_label":9,"correct_top1":1},{"video_id":"rhad_01930_head_stuck","true_label":6,"true_class":"head_stuck","predicted_label":6,"correct_top1":1},{"video_id":"rhad_00063_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00347_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00912_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_01331_collesion","true_label":3,"true_class":"collesion","predicted_label":3,"correct_top1":1},{"video_id":"rhad_00038_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00068_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02844_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_00039_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_00121_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02197_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_01079_climbing","true_label":2,"true_class":"climbing","predicted_label":2,"correct_top1":1},{"video_id":"rhad_02770_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01868_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_02066_normal","true_label":7,"true_class":"normal","predicted_label":7,"correct_top1":1},{"video_id":"rhad_02883_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1},{"video_id":"rhad_01852_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_00621_child_running","true_label":1,"true_class":"child_running","predicted_label":1,"correct_top1":1},{"video_id":"rhad_01678_falling","true_label":5,"true_class":"falling","predicted_label":8,"correct_top1":0},{"video_id":"rhad_01848_falling","true_label":5,"true_class":"falling","predicted_label":5,"correct_top1":1},{"video_id":"rhad_00037_Jumping","true_label":0,"true_class":"Jumping","predicted_label":0,"correct_top1":1},{"video_id":"rhad_02849_scribbling","true_label":10,"true_class":"scribbling","predicted_label":10,"correct_top1":1}]'''
)
embedded_validation_records = json.loads(
    r'''[{"video_id":"rhad_02358_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_02580_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01441_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_02140_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_02307_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01904_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01084_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_02235_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_02638_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01153_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01912_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_00150_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00229_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_03165_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_00264_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01835_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00327_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01640_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01033_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_03022_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_00326_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01805_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02865_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02270_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01915_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_01442_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_01380_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_01010_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00346_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00571_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02137_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_02862_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_00215_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_02763_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01083_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00213_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00708_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00548_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02076_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00653_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00574_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00497_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02073_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_02599_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01031_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01950_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_01439_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_01292_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01804_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01281_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_03174_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_01905_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00465_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00342_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01952_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_02309_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00711_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02234_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_03065_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_01914_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_02139_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_03115_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_00825_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01448_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_00341_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00826_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00228_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01923_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_00712_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02639_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02373_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00468_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01270_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01001_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01160_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01534_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00824_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00148_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_03176_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_00498_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02306_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00452_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01008_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00217_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00570_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02595_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02357_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01287_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01382_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_02579_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01011_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_02304_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_02581_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01003_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00301_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01864_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00652_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00214_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00081_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01907_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02507_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_00710_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02233_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01535_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00709_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01007_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01286_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01822_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01159_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_00299_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01444_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_01450_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_02710_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01285_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_02974_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02761_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01162_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_00547_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00080_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_02510_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_01151_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01837_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_03098_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_03112_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_01085_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01806_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00464_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00285_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00226_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00227_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01067_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_02765_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01646_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01066_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_02926_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_03020_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_02628_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02356_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00656_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01925_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_02271_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01030_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01897_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02485_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_01002_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_02508_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_01377_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_00147_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00467_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01164_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01149_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01006_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01949_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_02977_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01032_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00083_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00328_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_02641_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01082_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01074_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01839_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02706_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_00297_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01283_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_03019_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_01643_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02275_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00298_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_02142_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00266_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_03177_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_02072_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_02922_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02071_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_03076_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_00268_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00084_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01906_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01229_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_02377_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_02305_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01576_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00540_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02975_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_00466_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02441_objects_falling","true_label":8,"true_class":"objects_falling"},{"video_id":"rhad_02443_objects_falling","true_label":8,"true_class":"objects_falling"},{"video_id":"rhad_03121_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_02705_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01271_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01913_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_01425_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_02505_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_02596_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_03175_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_00296_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01379_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_02925_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_03087_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_02597_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02863_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01575_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02923_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01836_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01438_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_02074_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00655_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_03113_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_01951_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_02376_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00263_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_02273_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00082_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_02762_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01069_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00499_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02483_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_00216_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00823_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00572_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01884_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01642_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02760_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01886_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00286_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01096_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00546_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02375_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_02488_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_01579_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01887_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00225_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01885_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02374_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01273_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_02861_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01922_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_01275_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_02924_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02632_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_00290_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01436_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_00288_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01118_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_02484_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_01284_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_03143_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_01840_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01882_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01451_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_01150_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_03110_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_02630_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_00538_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01272_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01148_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_01447_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_00325_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01446_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_01009_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00324_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_00343_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_02708_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02707_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02272_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01924_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_02440_objects_falling","true_label":8,"true_class":"objects_falling"},{"video_id":"rhad_01954_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_02442_objects_falling","true_label":8,"true_class":"objects_falling"},{"video_id":"rhad_02486_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_02643_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01086_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00501_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02359_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_00151_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01577_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00287_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_02232_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01803_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02640_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_00536_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01443_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_03154_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_01065_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_02973_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02594_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02361_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_03173_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_02138_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01833_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_00535_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01926_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_03116_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_02506_pulling_curtains","true_label":9,"true_class":"pulling_curtains"},{"video_id":"rhad_00537_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01883_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01004_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00551_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_01378_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_01161_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_00569_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_03132_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_00500_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_00344_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_02629_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01440_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_01068_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_03018_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_01574_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01838_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_03017_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_02627_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01911_head_stuck","true_label":6,"true_class":"head_stuck"},{"video_id":"rhad_03114_sharp_objects","true_label":11,"true_class":"sharp_objects"},{"video_id":"rhad_01841_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02972_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_01802_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_02578_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_02439_objects_falling","true_label":8,"true_class":"objects_falling"},{"video_id":"rhad_01303_collesion","true_label":3,"true_class":"collesion"},{"video_id":"rhad_00549_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02236_normal","true_label":7,"true_class":"normal"},{"video_id":"rhad_01029_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01641_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01875_falling","true_label":5,"true_class":"falling"},{"video_id":"rhad_01081_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_02582_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_00654_child_running","true_label":1,"true_class":"child_running"},{"video_id":"rhad_02860_scribbling","true_label":10,"true_class":"scribbling"},{"video_id":"rhad_00265_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01449_electrical_appliances","true_label":4,"true_class":"electrical_appliances"},{"video_id":"rhad_00828_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_01107_climbing","true_label":2,"true_class":"climbing"},{"video_id":"rhad_00149_Jumping","true_label":0,"true_class":"Jumping"},{"video_id":"rhad_01843_falling","true_label":5,"true_class":"falling"}]'''
)

reference_test_df = pd.DataFrame(
    embedded_test_records
)
reference_validation_df = pd.DataFrame(
    embedded_validation_records
)

required_columns = {
    "video_id",
    "true_label",
    "true_class",
}

for name, dataframe in [
    ("test", reference_test_df),
    ("validation", reference_validation_df),
]:
    assert required_columns.issubset(dataframe.columns), (
        f"Colonnes manquantes dans le CSV {name} : "
        f"{required_columns - set(dataframe.columns)}"
    )

assert len(reference_test_df) == 335
assert len(reference_validation_df) == 335

# Vérifier que le mapping historique correspond au mapping courant.
historical_mapping_df = pd.concat(
    [
        reference_test_df[
            ["true_label", "true_class"]
        ],
        reference_validation_df[
            ["true_label", "true_class"]
        ],
    ],
    ignore_index=True,
).drop_duplicates()

for _, row in historical_mapping_df.iterrows():
    assert class_to_idx[str(row["true_class"])] == int(
        row["true_label"]
    ), (
        "Le mapping des classes diffère du mapping historique : "
        f"{row.to_dict()}"
    )

# ------------------------------------------------------------
# 2. Séparer les 3248 anciennes vidéos des 55 nouvelles
# ------------------------------------------------------------

all_videos = sorted(
    path
    for path in TRAINING2_ROOT.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower() in VIDEO_EXTENSIONS
    )
)

assert len(all_videos) == 3303, (
    f"3303 vidéos attendues, trouvé : {len(all_videos)}"
)

added_pattern = re.compile(
    r"^1 \((\d+)\)\.mp4$",
    re.IGNORECASE,
)

added_videos = []

for path in all_videos:
    match = added_pattern.match(path.name)

    if (
        path.parent.name == "objects_falling"
        and match
        and 61 <= int(match.group(1)) <= 115
    ):
        added_videos.append(path)

assert len(added_videos) == 55, (
    f"55 nouvelles vidéos attendues, trouvé : "
    f"{len(added_videos)}"
)

added_set = set(added_videos)

old_videos = [
    path
    for path in all_videos
    if path not in added_set
]

assert len(old_videos) == 3248

# ------------------------------------------------------------
# 3. Recréer le même identifiant rhad_XXXXX_classe
# ------------------------------------------------------------

# Les anciens identifiants ont été attribués par ordre de classe,
# puis par ordre lexicographique des fichiers dans chaque classe.
alias_to_source = {}
source_to_alias = {}
global_index = 0

for class_name in CLASS_NAMES:
    class_old_videos = sorted(
        path
        for path in old_videos
        if path.parent.name == class_name
    )

    for source_path in class_old_videos:
        alias_stem = (
            f"rhad_{global_index:05d}_{class_name}"
        )

        alias_to_source[alias_stem] = source_path
        source_to_alias[source_path] = (
            alias_stem + source_path.suffix.lower()
        )

        global_index += 1

assert global_index == 3248
assert len(alias_to_source) == 3248

historical_test_ids = set(
    reference_test_df["video_id"].astype(str)
)

historical_validation_ids = set(
    reference_validation_df["video_id"].astype(str)
)

assert not (
    historical_test_ids
    & historical_validation_ids
), "Le test et la validation historiques se chevauchent."

missing_test_ids = (
    historical_test_ids
    - set(alias_to_source)
)

missing_validation_ids = (
    historical_validation_ids
    - set(alias_to_source)
)

assert not missing_test_ids, (
    "Identifiants du test impossibles à relier aux vidéos : "
    f"{sorted(missing_test_ids)[:10]}"
)

assert not missing_validation_ids, (
    "Identifiants de validation impossibles à relier aux vidéos : "
    f"{sorted(missing_validation_ids)[:10]}"
)

test_samples = [
    alias_to_source[str(video_id)]
    for video_id in reference_test_df["video_id"]
]

validation_samples = [
    alias_to_source[str(video_id)]
    for video_id in reference_validation_df["video_id"]
]

evaluation_set = (
    set(test_samples)
    | set(validation_samples)
)

train_old_samples = [
    path
    for path in old_videos
    if path not in evaluation_set
]

train_samples = (
    train_old_samples
    + added_videos
)

assert len(train_old_samples) == 2578
assert len(train_samples) == 2633
assert len(validation_samples) == 335
assert len(test_samples) == 335

assert not (
    set(train_samples)
    & set(validation_samples)
)

assert not (
    set(train_samples)
    & set(test_samples)
)

assert not (
    set(validation_samples)
    & set(test_samples)
)

assert added_set.issubset(set(train_samples))

# ------------------------------------------------------------
# 4. Créer les alias vidéo
# ------------------------------------------------------------

sample_to_alias = {}

for source_path, alias_name in source_to_alias.items():
    alias_path = VIDEO_ROOT / alias_name
    os.symlink(str(source_path), str(alias_path))
    sample_to_alias[source_path] = alias_name

for index, source_path in enumerate(
    sorted(added_videos)
):
    alias_name = (
        f"rhad_new_{index:03d}_objects_falling"
        f"{source_path.suffix.lower()}"
    )

    alias_path = VIDEO_ROOT / alias_name
    os.symlink(str(source_path), str(alias_path))
    sample_to_alias[source_path] = alias_name

assert len(sample_to_alias) == 3303

# ------------------------------------------------------------
# 5. Écrire les annotations
# ------------------------------------------------------------

def write_training_annotations(
    output_path: Path,
    samples,
):
    with output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for source_path in samples:
            class_name = source_path.parent.name
            label = class_to_idx[class_name]
            file.write(
                f"{sample_to_alias[source_path]} "
                f"{label}\n"
            )


def write_reference_annotations(
    output_path: Path,
    reference_dataframe: pd.DataFrame,
):
    with output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for _, row in reference_dataframe.iterrows():
            source_path = alias_to_source[
                str(row["video_id"])
            ]

            file.write(
                f"{sample_to_alias[source_path]} "
                f"{int(row['true_label'])}\n"
            )


write_training_annotations(
    ANNOTATIONS_DIR / "train.csv",
    train_samples,
)

write_reference_annotations(
    ANNOTATIONS_DIR / "val.csv",
    reference_validation_df,
)

write_reference_annotations(
    ANNOTATIONS_DIR / "test.csv",
    reference_test_df,
)

with (
    ANNOTATIONS_DIR / "label_mapping.csv"
).open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.writer(file)
    writer.writerow(["label", "class_name"])

    for class_name, label in class_to_idx.items():
        writer.writerow([label, class_name])

summary = {
    "split_source": (
        "historical VideoMAE prediction CSV identifiers"
    ),
    "training2_total": 3303,
    "old_videos": 3248,
    "new_videos": 55,
    "train": 2633,
    "validation": 335,
    "test": 335,
    "class_names": CLASS_NAMES,
    "reference_test_source": "embedded historical manifest",
    "reference_validation_source": "embedded historical manifest",
}

(
    ANNOTATIONS_DIR / "split_summary.json"
).write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("=" * 70)
print("SPLIT HISTORIQUE EXACT PRÊT")
print("=" * 70)
print("Train      :", len(train_samples))
print("Validation :", len(validation_samples))
print("Test       :", len(test_samples))
print(
    "Support objects_falling dans le test :",
    int(
        (
            reference_test_df["true_class"]
            == "objects_falling"
        ).sum()
    ),
)
print(
    "Accuracy ancien CSV de référence :",
    float(
        reference_test_df[
            "correct_top1"
        ].mean()
    ),
)
print("Nouvelles vidéos dans test : 0")
print("Fuites : 0")

assert int(
    (
        reference_test_df["true_class"]
        == "objects_falling"
    ).sum()
) == 5

assert abs(
    float(
        reference_test_df[
            "correct_top1"
        ].mean()
    )
    - 0.8865671641791045
) < 1e-9, (
    "Le CSV de référence n'est pas le test "
    "standardisé attendu (88.657 %)."
)


In [ ]:
# ============================================================
# PRÉPARER LES DOSSIERS D'ÉVALUATION
# ============================================================

from pathlib import Path
import shutil
import pandas as pd

PY = "/kaggle/working/vmae_env/bin/python"
REPO_DIR = Path("/kaggle/working/VideoMAEv2")
RUN_FILE = REPO_DIR / "run_class_finetuning.py"

COMPARISON_DIR = Path(
    "/kaggle/working/videomae_old_new_comparison"
)

TEST_ANNOTATIONS_DIR = (
    COMPARISON_DIR / "annotations_test"
)

if COMPARISON_DIR.exists():
    shutil.rmtree(COMPARISON_DIR)

TEST_ANNOTATIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# run_class_finetuning attend les trois fichiers
shutil.copy2(
    ANNOTATIONS_DIR / "train.csv",
    TEST_ANNOTATIONS_DIR / "train.csv",
)
shutil.copy2(
    ANNOTATIONS_DIR / "val.csv",
    TEST_ANNOTATIONS_DIR / "val.csv",
)
shutil.copy2(
    ANNOTATIONS_DIR / "test.csv",
    TEST_ANNOTATIONS_DIR / "test.csv",
)

test_df = pd.read_csv(
    TEST_ANNOTATIONS_DIR / "test.csv",
    header=None,
)

assert len(test_df) == 335

for path in [
    Path(PY),
    RUN_FILE,
    VIDEO_ROOT,
    OLD_CHECKPOINT,
    NEW_CHECKPOINT,
]:
    assert path.exists(), path

print("Évaluation préparée sur", len(test_df), "vidéos")
print("Chaque modèle produira 335 × 2 × 3 = 2010 vues")


In [ ]:
# ============================================================
# ÉVALUER LES DEUX CHECKPOINTS SUR LE MÊME TEST
# ============================================================

from pathlib import Path
import datetime
import os
import shutil
import subprocess

def evaluate_checkpoint(
    checkpoint_path: Path,
    model_name: str,
    master_port: int,
):
    output_dir = COMPARISON_DIR / f"{model_name}_raw"

    if output_dir.exists():
        shutil.rmtree(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    command = [
        PY,
        "-m",
        "torch.distributed.run",
        "--nproc_per_node=2",
        f"--master_port={master_port}",
        "run_class_finetuning.py",

        "--model", "vit_base_patch16_224",
        "--nb_classes", "12",
        "--use_mean_pooling",
        "--init_scale", "0.001",

        "--data_set", "RHAD",
        "--data_path", str(TEST_ANNOTATIONS_DIR),
        "--data_root", str(VIDEO_ROOT),

        "--input_size", "224",
        "--short_side_size", "224",
        "--num_frames", "16",
        "--num_segments", "1",
        "--tubelet_size", "2",
        "--sampling_rate", "4",

        "--batch_size", "2",
        "--num_workers", "2",
        "--seed", "42",
        "--test_num_segment", "2",
        "--test_num_crop", "3",

        "--finetune", str(checkpoint_path),
        "--model_key", "model",
        "--no_auto_resume",
        "--eval",

        "--output_dir", str(output_dir),
        "--log_dir", str(output_dir),
    ]

    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = "0,1"
    environment["OMP_NUM_THREADS"] = "1"
    environment["PYTHONUNBUFFERED"] = "1"
    environment["TOKENIZERS_PARALLELISM"] = "false"

    log_path = COMPARISON_DIR / f"{model_name}_console.log"

    print("\n" + "=" * 70)
    print("ÉVALUATION :", model_name)
    print("=" * 70)
    print("Checkpoint :", checkpoint_path)
    print("Sortie     :", output_dir)

    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=str(REPO_DIR),
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None

        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Échec de l'évaluation {model_name}, code {return_code}"
        )

    assert (output_dir / "0.txt").exists()
    assert (output_dir / "1.txt").exists()

    return output_dir

OLD_RAW_DIR = evaluate_checkpoint(
    OLD_CHECKPOINT,
    "ancien",
    29541,
)

NEW_RAW_DIR = evaluate_checkpoint(
    NEW_CHECKPOINT,
    "nouveau",
    29542,
)

print("\nLES DEUX ÉVALUATIONS SONT TERMINÉES")


In [ ]:
# ============================================================
# FUSIONNER LES 6 VUES ET CALCULER TOUTES LES MÉTRIQUES
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


# ============================================================
# CLASSES
# ============================================================

CLASS_NAMES = [
    row.strip().split(",", 1)[1]
    for row in (
        ANNOTATIONS_DIR
        / "label_mapping.csv"
    ).read_text(
        encoding="utf-8"
    ).splitlines()[1:]
]

NUM_CLASSES = len(CLASS_NAMES)

assert NUM_CLASSES == 12, (
    f"12 classes attendues, trouvé : {NUM_CLASSES}"
)

assert "objects_falling" in CLASS_NAMES

OBJECTS_FALLING_INDEX = (
    CLASS_NAMES.index(
        "objects_falling"
    )
)

print("Classes :", CLASS_NAMES)


# ============================================================
# FONCTIONS UTILES
# ============================================================

def normalize_video_id(value):
    return Path(
        str(value).strip()
    ).stem


def stable_softmax(logits):
    logits = np.asarray(
        logits,
        dtype=np.float64,
    )

    logits = logits - np.max(logits)

    values = np.exp(logits)

    return values / values.sum()


# ============================================================
# LIRE LES PRÉDICTIONS BRUTES VIDEOMAE
# ============================================================

def read_raw_predictions(
    raw_directory: Path,
):

    rows = []

    raw_files = [
        raw_directory / "0.txt",
        raw_directory / "1.txt",
    ]

    for raw_file in raw_files:

        assert raw_file.exists(), (
            f"Fichier introuvable : {raw_file}"
        )

        lines = raw_file.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()

        for line in lines:

            if (
                "[" not in line
                or "]" not in line
            ):
                continue

            video_part, remaining = (
                line.split(
                    "[",
                    1,
                )
            )

            logits_text, metadata_text = (
                remaining.split(
                    "]",
                    1,
                )
            )

            logits = np.fromstring(
                logits_text,
                dtype=np.float64,
                sep=",",
            )

            metadata = (
                metadata_text
                .strip()
                .split()
            )

            if len(logits) != NUM_CLASSES:
                continue

            if len(metadata) < 3:
                continue

            row = {
                "video_id":
                    normalize_video_id(
                        video_part
                    ),

                "true_label":
                    int(metadata[0]),

                "segment_index":
                    int(metadata[1]),

                "crop_index":
                    int(metadata[2]),
            }

            for index in range(
                NUM_CLASSES
            ):
                row[
                    f"logit_{index}"
                ] = float(
                    logits[index]
                )

            rows.append(row)

    dataframe = pd.DataFrame(
        rows
    )

    print(
        raw_directory.name,
        "->",
        len(dataframe),
        "vues trouvées",
    )

    assert len(dataframe) == 2010, (
        f"{len(dataframe)} vues trouvées "
        "au lieu de 2010."
    )

    return dataframe


# ============================================================
# CALCULER LES RÉSULTATS D'UN CHECKPOINT
# ============================================================

def build_results(
    raw_directory: Path,
    model_name: str,
):

    raw_df = read_raw_predictions(
        raw_directory
    )

    annotations_df = pd.read_csv(
        TEST_ANNOTATIONS_DIR
        / "test.csv",
        header=None,
        sep=r"\s+",
        names=[
            "video_path",
            "true_label",
        ],
        engine="python",
    )

    print(
        model_name,
        "-> annotations test :",
        len(annotations_df),
    )

    assert len(annotations_df) == 335, (
        f"335 vidéos attendues, trouvé : "
        f"{len(annotations_df)}"
    )

    assert annotations_df[
        "true_label"
    ].notna().all(), (
        "Certains labels du test sont vides."
    )

    annotations_df[
        "true_label"
    ] = annotations_df[
        "true_label"
    ].astype(int)

    annotations_df[
        "video_id"
    ] = annotations_df[
        "video_path"
    ].map(
        normalize_video_id
    )

    rows = []

    logit_columns = [
        f"logit_{index}"
        for index in range(
            NUM_CLASSES
        )
    ]

    for _, annotation in (
        annotations_df.iterrows()
    ):

        video_id = annotation[
            "video_id"
        ]

        true_label = int(
            annotation[
                "true_label"
            ]
        )

        views = raw_df[
            raw_df["video_id"]
            == video_id
        ]

        assert len(views) == 6, (
            f"{video_id} : "
            f"{len(views)} vues au lieu de 6."
        )

        mean_logits = (
            views[
                logit_columns
            ]
            .to_numpy(
                dtype=np.float64
            )
            .mean(axis=0)
        )

        probabilities = stable_softmax(
            mean_logits
        )

        predicted_label = int(
            np.argmax(
                probabilities
            )
        )

        top5 = np.argsort(
            -probabilities
        )[:5]

        row = {
            "model":
                model_name,

            "video_id":
                video_id,

            "video_path":
                str(
                    annotation[
                        "video_path"
                    ]
                ),

            "true_label":
                true_label,

            "true_class":
                CLASS_NAMES[
                    true_label
                ],

            "predicted_label":
                predicted_label,

            "predicted_class":
                CLASS_NAMES[
                    predicted_label
                ],

            "confidence":
                float(
                    probabilities[
                        predicted_label
                    ]
                ),

            "correct_top1":
                int(
                    predicted_label
                    == true_label
                ),

            "correct_top5":
                int(
                    true_label
                    in top5
                ),
        }

        for index in range(
            NUM_CLASSES
        ):
            row[
                f"prob_{index}"
            ] = float(
                probabilities[index]
            )

        rows.append(row)

    predictions = pd.DataFrame(
        rows
    )

    assert len(predictions) == 335

    y_true = predictions[
        "true_label"
    ].to_numpy(
        dtype=int
    )

    y_pred = predictions[
        "predicted_label"
    ].to_numpy(
        dtype=int
    )

    report_dict = (
        classification_report(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        )
    )

    report_text = (
        classification_report(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0,
        )
    )

    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
    )

    metrics = {
        "Version":
            model_name,

        "Accuracy_Top1":
            float(
                accuracy_score(
                    y_true,
                    y_pred,
                )
            ),

        "Accuracy_Top5":
            float(
                predictions[
                    "correct_top5"
                ].mean()
            ),

        "Macro_Precision":
            float(
                precision_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0,
                )
            ),

        "Macro_Recall":
            float(
                recall_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0,
                )
            ),

        "Macro_F1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0,
                )
            ),

        "Weighted_F1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0,
                )
            ),

        "objects_falling_precision":
            float(
                report_dict[
                    "objects_falling"
                ]["precision"]
            ),

        "objects_falling_recall":
            float(
                report_dict[
                    "objects_falling"
                ]["recall"]
            ),

        "objects_falling_f1":
            float(
                report_dict[
                    "objects_falling"
                ]["f1-score"]
            ),

        "objects_falling_support":
            int(
                report_dict[
                    "objects_falling"
                ]["support"]
            ),

        "errors_total":
            int(
                (
                    y_true
                    != y_pred
                ).sum()
            ),
    }

    model_dir = (
        COMPARISON_DIR
        / model_name
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    predictions.to_csv(
        model_dir
        / "test_predictions.csv",
        index=False,
    )

    (
        model_dir
        / "classification_report.txt"
    ).write_text(
        report_text,
        encoding="utf-8",
    )

    (
        model_dir
        / "classification_report.json"
    ).write_text(
        json.dumps(
            report_dict,
            indent=2,
        ),
        encoding="utf-8",
    )

    pd.DataFrame(
        confusion,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    ).to_csv(
        model_dir
        / "confusion_matrix.csv"
    )

    (
        model_dir
        / "metrics.json"
    ).write_text(
        json.dumps(
            metrics,
            indent=2,
        ),
        encoding="utf-8",
    )

    return (
        predictions,
        metrics,
        confusion,
        report_dict,
    )


# ============================================================
# ANCIEN CHECKPOINT
# ============================================================

(
    old_predictions,
    old_metrics,
    old_cm,
    old_report,
) = build_results(
    OLD_RAW_DIR,
    "Ancien checkpoint",
)


# ============================================================
# CONTRÔLE BLOQUANT DU SPLIT HISTORIQUE
# ============================================================

# Si les alias ont été reliés aux mauvaises vidéos, l'ancien checkpoint
# ne reproduira pas les prédictions archivées. La comparaison doit alors
# s'arrêter au lieu de produire des métriques trompeuses.
reference_old_predictions = (
    reference_test_df[
        [
            "video_id",
            "true_label",
            "predicted_label",
        ]
    ]
    .copy()
)

reference_old_predictions[
    "video_id"
] = reference_old_predictions[
    "video_id"
].map(normalize_video_id)

split_verification = old_predictions[
    [
        "video_id",
        "true_label",
        "predicted_label",
    ]
].merge(
    reference_old_predictions,
    on="video_id",
    how="inner",
    suffixes=(
        "_reevalue",
        "_reference",
    ),
)

assert len(split_verification) == 335, (
    "Impossible d'apparier les 335 prédictions historiques."
)

label_agreement = float(
    (
        split_verification[
            "true_label_reevalue"
        ]
        == split_verification[
            "true_label_reference"
        ]
    ).mean()
)

prediction_agreement = float(
    (
        split_verification[
            "predicted_label_reevalue"
        ]
        == split_verification[
            "predicted_label_reference"
        ]
    ).mean()
)

print(
    "Accord des labels avec le test historique :",
    f"{label_agreement:.2%}",
)
print(
    "Accord des prédictions de l'ancien checkpoint :",
    f"{prediction_agreement:.2%}",
)

assert label_agreement == 1.0
assert prediction_agreement == 1.0, (
    "Le split reconstruit ne reproduit pas exactement les anciennes "
    "prédictions. Ne pas interpréter la comparaison : il faut retrouver "
    "le manifeste historique reliant les identifiants aux fichiers."
)

print(
    "Split historique vérifié : l'ancien checkpoint reproduit "
    "exactement les 335 prédictions archivées."
)


# ============================================================
# NOUVEAU CHECKPOINT
# ============================================================

(
    new_predictions,
    new_metrics,
    new_cm,
    new_report,
) = build_results(
    NEW_RAW_DIR,
    "Nouveau checkpoint",
)


# ============================================================
# TABLEAU FINAL
# ============================================================

comparison_df = pd.DataFrame(
    [
        old_metrics,
        new_metrics,
    ]
)

comparison_df.to_csv(
    COMPARISON_DIR
    / "comparison_metrics.csv",
    index=False,
)

print("\n" + "=" * 70)
print("COMPARAISON ANCIEN / NOUVEAU VIDEOMAE")
print("=" * 70)

display(
    comparison_df.round(4)
)

In [ ]:
# ============================================================
# MATRICES DE CONFUSION : BRUTES ET NORMALISÉES
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

def plot_confusion(
    confusion,
    title,
    output_path,
    normalize=False,
):
    matrix = confusion.astype(np.float64)

    if normalize:
        row_sums = matrix.sum(axis=1, keepdims=True)
        matrix = np.divide(
            matrix,
            row_sums,
            out=np.zeros_like(matrix),
            where=row_sums != 0,
        )

    fig, ax = plt.subplots(figsize=(14, 12))
    image = ax.imshow(matrix)

    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(
        CLASS_NAMES,
        rotation=90,
    )
    ax.set_yticklabels(CLASS_NAMES)

    ax.set_xlabel("Classe prédite")
    ax.set_ylabel("Classe réelle")
    ax.set_title(title)

    threshold = matrix.max() / 2 if matrix.size else 0

    for row in range(NUM_CLASSES):
        for column in range(NUM_CLASSES):
            value = matrix[row, column]

            text = (
                f"{value:.2f}"
                if normalize
                else str(int(value))
            )

            ax.text(
                column,
                row,
                text,
                ha="center",
                va="center",
                color=(
                    "white"
                    if value > threshold
                    else "black"
                ),
                fontsize=8,
            )

    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

plot_confusion(
    old_cm,
    "Matrice de confusion — Ancien VideoMAE",
    COMPARISON_DIR / "confusion_matrix_old.png",
    normalize=False,
)

plot_confusion(
    new_cm,
    "Matrice de confusion — Nouveau VideoMAE",
    COMPARISON_DIR / "confusion_matrix_new.png",
    normalize=False,
)

plot_confusion(
    old_cm,
    "Matrice normalisée — Ancien VideoMAE",
    COMPARISON_DIR / "confusion_matrix_old_normalized.png",
    normalize=True,
)

plot_confusion(
    new_cm,
    "Matrice normalisée — Nouveau VideoMAE",
    COMPARISON_DIR / "confusion_matrix_new_normalized.png",
    normalize=True,
)


In [ ]:
# ============================================================
# COMPARAISON PAR CLASSE + DIFFÉRENCES
# ============================================================

import pandas as pd

rows = []

for class_name in CLASS_NAMES:
    old_values = old_report[class_name]
    new_values = new_report[class_name]

    rows.append(
        {
            "Classe": class_name,
            "Ancien_precision": old_values["precision"],
            "Nouveau_precision": new_values["precision"],
            "Delta_precision": (
                new_values["precision"]
                - old_values["precision"]
            ),
            "Ancien_recall": old_values["recall"],
            "Nouveau_recall": new_values["recall"],
            "Delta_recall": (
                new_values["recall"]
                - old_values["recall"]
            ),
            "Ancien_F1": old_values["f1-score"],
            "Nouveau_F1": new_values["f1-score"],
            "Delta_F1": (
                new_values["f1-score"]
                - old_values["f1-score"]
            ),
            "Support": int(old_values["support"]),
        }
    )

per_class_df = pd.DataFrame(rows)

per_class_df.to_csv(
    COMPARISON_DIR / "comparison_per_class.csv",
    index=False,
)

display(
    per_class_df.round(4)
)

print("\n" + "=" * 70)
print("RÉSUMÉ OBJECTS_FALLING")
print("=" * 70)

display(
    per_class_df[
        per_class_df["Classe"] == "objects_falling"
    ].round(4)
)

print("\n" + "=" * 70)
print("DIFFÉRENCE GLOBALE NOUVEAU - ANCIEN")
print("=" * 70)

for metric in [
    "Accuracy_Top1",
    "Accuracy_Top5",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Weighted_F1",
    "objects_falling_precision",
    "objects_falling_recall",
    "objects_falling_f1",
]:
    difference = (
        new_metrics[metric]
        - old_metrics[metric]
    )

    sign = "+" if difference >= 0 else ""

    print(
        f"{metric:30s}: "
        f"{sign}{difference:.4f}"
    )


In [ ]:
# ============================================================
# VIDÉOS DONT LA PRÉDICTION A CHANGÉ
# ============================================================

merged = old_predictions[
    [
        "video_id",
        "video_path",
        "true_class",
        "predicted_class",
        "correct_top1",
        "confidence",
    ]
].merge(
    new_predictions[
        [
            "video_id",
            "predicted_class",
            "correct_top1",
            "confidence",
        ]
    ],
    on="video_id",
    suffixes=("_ancien", "_nouveau"),
)

merged["changement"] = (
    merged["predicted_class_ancien"]
    != merged["predicted_class_nouveau"]
)

merged["corrige_par_nouveau"] = (
    (merged["correct_top1_ancien"] == 0)
    & (merged["correct_top1_nouveau"] == 1)
)

merged["degrade_par_nouveau"] = (
    (merged["correct_top1_ancien"] == 1)
    & (merged["correct_top1_nouveau"] == 0)
)

merged.to_csv(
    COMPARISON_DIR / "prediction_changes.csv",
    index=False,
)

print(
    "Prédictions changées :",
    int(merged["changement"].sum()),
)

print(
    "Erreurs corrigées par le nouveau :",
    int(merged["corrige_par_nouveau"].sum()),
)

print(
    "Bonnes prédictions dégradées :",
    int(merged["degrade_par_nouveau"].sum()),
)

display(
    merged[
        merged["changement"]
    ].head(30)
)


In [ ]:
# ============================================================
# VÉRIFIER QUE LES DEUX CHECKPOINTS SONT DIFFÉRENTS
# ============================================================

import hashlib
from pathlib import Path
import torch


def sha256_file(path: Path) -> str:
    sha = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            block = file.read(1024 * 1024)

            if not block:
                break

            sha.update(block)

    return sha.hexdigest()


print("Ancien :", OLD_CHECKPOINT)
print("Nouveau :", NEW_CHECKPOINT)

print("\nSHA256 ancien :")
print(sha256_file(OLD_CHECKPOINT))

print("\nSHA256 nouveau :")
print(sha256_file(NEW_CHECKPOINT))


old_checkpoint = torch.load(
    OLD_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

new_checkpoint = torch.load(
    NEW_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

old_state = old_checkpoint["model"]
new_state = new_checkpoint["model"]

different_tensors = 0
maximum_difference = 0.0
total_difference = 0.0

for key in old_state:

    assert key in new_state, key

    old_tensor = old_state[key].float()
    new_tensor = new_state[key].float()

    difference = (
        old_tensor - new_tensor
    ).abs()

    current_maximum = float(
        difference.max()
    )

    current_total = float(
        difference.sum()
    )

    if current_maximum > 0:
        different_tensors += 1

    maximum_difference = max(
        maximum_difference,
        current_maximum,
    )

    total_difference += current_total


print("\nNombre de tenseurs différents :", different_tensors)
print("Différence maximale           :", maximum_difference)
print("Différence totale             :", total_difference)

assert sha256_file(OLD_CHECKPOINT) != sha256_file(
    NEW_CHECKPOINT
), (
    "ATTENTION : les deux fichiers sont identiques."
)

assert different_tensors > 0, (
    "ATTENTION : les poids des deux modèles sont identiques."
)

print("\nLes deux checkpoints sont réellement différents.")

In [ ]:
# ============================================================
# CRÉER UNE ARCHIVE ZIP AVEC TOUS LES RÉSULTATS
# ============================================================

from pathlib import Path
import shutil

# Conserver les manifestes historiques comme références.
reference_test_df.to_csv(
    COMPARISON_DIR
    / "reference_old_test_predictions.csv",
    index=False,
)

reference_validation_df.to_csv(
    COMPARISON_DIR
    / "reference_old_validation_predictions.csv",
    index=False,
)

archive_path = shutil.make_archive(
    "/kaggle/working/videomae_old_new_comparison",
    "zip",
    root_dir=str(COMPARISON_DIR),
)

print("=" * 70)
print("COMPARAISON TERMINÉE")
print("=" * 70)
print("Dossier :", COMPARISON_DIR)
print("Archive :", archive_path)

print("\nFichiers principaux :")
print(
    COMPARISON_DIR / "comparison_metrics.csv"
)
print(
    COMPARISON_DIR / "comparison_per_class.csv"
)
print(
    COMPARISON_DIR / "confusion_matrix_old.png"
)
print(
    COMPARISON_DIR / "confusion_matrix_new.png"
)
print(
    COMPARISON_DIR / "prediction_changes.csv"
)
print(
    COMPARISON_DIR
    / "reference_old_test_predictions.csv"
)
